## Load Data


In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from pathlib import Path

### Loading Data

In [ ]:

# Load the full dataset
dataset = load_dataset("divarofficial/real_estate_ads")

# Get dataset statistics
print(f"Dataset size: {len(dataset['train'])} rows")
print(f"Features: {dataset['train'].features}")

### Convert to Data Frame

In [ ]:
df = dataset["train"].to_pandas()
df.head(5)

In [ ]:
df.info()
df.columns
df.describe()

## Cleaning Data

In [ ]:
mapping = {
    "null" : np.nan,
    "بدون اتاق":0,
    "یک": 1,
    "دو": 2,
    "سه": 3,
    "چهار": 4,
    "پنج یا بیشتر" : 5
}

df["rooms_count"] = df["rooms_count"].replace(mapping)



# ------------------------------------------------------------------------------


construction_year_mapping = {
    "null": np.nan,
    None: np.nan,
    
    "قبل از ۱۳۷۰": 1370,
    "۱۳۷۱": 1371,
    "۱۳۷۲": 1372,
    "۱۳۷۳": 1373,
    "۱۳۷۴": 1374,
    "۱۳۷۵": 1375,
    "۱۳۷۶": 1376,
    "۱۳۷۷": 1377,
    "۱۳۷۸": 1378,
    "۱۳۷۹": 1379,
    "۱۳۸۰": 1380,
    "۱۳۸۱": 1381,
    "۱۳۸۲": 1382,
    "۱۳۸۳": 1383,
    "۱۳۸۴": 1384,
    "۱۳۸۵": 1385,
    "۱۳۸۶": 1386,
    "۱۳۸۷": 1387,
    "۱۳۸۸": 1388,
    "۱۳۸۹": 1389,
    "۱۳۹۰": 1390,
    "۱۳۹۱": 1391,
    "۱۳۹۲": 1392,
    "۱۳۹۳": 1393,
    "۱۳۹۴": 1394,
    "۱۳۹۵": 1395,
    "۱۳۹۶": 1396,
    "۱۳۹۷": 1397,
    "۱۳۹۸": 1398,
    "۱۳۹۹": 1399,
    "۱۴۰۰": 1400,
    "۱۴۰۱": 1401,
    "۱۴۰۲": 1402,
    "۱۴۰۳": 1403,
}

df["construction_year"] = df["construction_year"].replace(construction_year_mapping).astype("Int16")


# Change created_at_month type to datetime
df["created_at_month"] = pd.to_datetime(
    df["created_at_month"].astype("string"),
    errors="coerce"
)


In [ ]:
df["floor"]=(
    df["floor"].replace({"30+":"31"}).astype("Int8")
)

df["total_floors_count"]=(
    df["total_floors_count"].replace({"30+":"31"}).replace("unselect",None).astype("Int8")
)
# print(df["total_floors_count"].unique())

df["unit_per_floor"]=(
    df["unit_per_floor"].replace({"more_than_8":"9"}).replace("unselect",None).astype("Int8")
)

df["extra_person_capacity"]=(
    df["extra_person_capacity"].replace({"30+":"31"}).astype("Int8")
)

df['rooms_count'] = df['rooms_count'].astype("Int8")

df["regular_person_capacity"] = df["regular_person_capacity"].astype("Int8")

# has only 3 value nan.300,500
df["location_radius"] = df["location_radius"].astype("Int16") 

df["land_size"] = df["land_size"].astype("Int32")



print(df["floor"].unique())


In [ ]:
df["transformable_price"] = df["transformable_price"].astype("boolean")

df["rent_credit_transform"] = df["rent_credit_transform"].astype("boolean")

df["has_business_deed"] = df["has_business_deed"].astype("boolean")

In [ ]:
print(df["floor"].unique())

In [ ]:
df["has_balcony"] = df["has_balcony"].replace("unselect",None).replace("true",True).replace("false",False).astype("boolean")
df["has_elevator"] = df["has_elevator"].astype("boolean")
df["has_warehouse"] = df["has_warehouse"].astype("boolean")
df["has_parking"] = df["has_parking"].astype("boolean")
df["is_rebuilt"] = df["is_rebuilt"].astype("boolean")
df["has_water"] = df["has_water"].astype("boolean")
df["has_electricity"] = df["has_electricity"].astype("boolean")
df["has_gas"] = df["has_gas"].astype("boolean")
df["has_pool"] = df["has_pool"].astype("boolean")
df["has_jacuzzi"] = df["has_jacuzzi"].astype("boolean")
df["has_sauna"] = df["has_sauna"].astype("boolean")
df["has_security_guard"] = df["has_security_guard"].astype("boolean")
df["has_barbecue"] = df["has_barbecue"].astype("boolean")
df['rent_to_single'] = df['rent_to_single'].astype('boolean')


In [ ]:
df["has_restroom"] = df["has_restroom"].replace("unselect",None)
df["has_cooling_system"] = df["has_cooling_system"].replace("unselect",None)
df["has_warm_water_provider"] = df["has_warm_water_provider"].replace("unselect",None)
df["deed_type"] = df["deed_type"].replace("unselect",None)

In [ ]:
df.select_dtypes(include=["object"]).columns

Convert columns to Category to Reduce file size

In [ ]:
category_cols = [
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "neighborhood_slug",
    "user_type",
    "rent_mode",
    "rent_type",
    "price_mode",
    "credit_mode",
    "deed_type",
    "property_type",
    "building_direction",
    "floor_material",
    "has_restroom",
    "has_warm_water_provider",
    "has_heating_system",
    "has_cooling_system"
]
for col in category_cols:
    df[col]= df[col].astype("category")

In [ ]:
df.info()

In [ ]:
# لیست ستون‌های اعشاری (float64)
float_cols = df.select_dtypes(include=['float64']).columns
print("Float64 Columns:")
print(float_cols)


### Checking the percentage and number of missing values

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean()*100
})

missing = missing.sort_values(
    "missing_percent",
    ascending=False
)

missing.head(60)

In [ ]:
for col in df.columns:
    if col in ['title', 'description']:
        continue

    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"\n{col} -> min: {df[col].min()} | max: {df[col].max()}")
    else:
        print(f"\n{col} -> {df[col].unique()}")



In [ ]:
# Clean Persian Text

def normalize_text(text):
    if not isinstance(text, str):
        return text
    
    # ۱. جایگزینی جامع و قطعی اعداد فارسی و عربی به انگلیسی
    # این نقشه شامل هم اعداد فارسی (۰-۹) و هم اعداد عربی (٠-٩) است
    arabic_persian_digits = {
        '۰': '0', '۱': '1', '۲': '2', '۳': '3', '۴': '4', '۵': '5', '۶': '6', '۷': '7', '۸': '8', '۹': '9',
        '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9'
    }
    
    # تبدیل سریع کاراکتر به کاراکتر برای اعداد
    text = text.translate(str.maketrans(arabic_persian_digits))
    
    # ۲. اصلاح حروف عربی به فارسی
    clean_chars = {
        'ك': 'ک', 
        'ي': 'ی', 
        'ئ': 'ی',
        'ة': 'ه'
    }
    text = text.translate(str.maketrans(clean_chars))
    
    # ۳. تبدیل نیم‌فاصله (\u200c) به فاصله ساده
    text = text.replace('\u200c', ' ')
    
    # ۴. حذف فاصله‌های اضافی و فضاهای خالی متوالی
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# اعمال روی ستون‌های متنی (به خصوص ستون‌های سنگین title و description)
text_cols = ["title", "description"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).map(normalize_text)

print("نرمال‌سازی با موفقیت روی ستون‌ها اعمال شد.")


## save data to use for next level

In [ ]:

output_path = Path("../Outputs/01_df.feather")

output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_feather(output_path)
